# train 폴드 외 ChemBERTa 채점 (Colab GPU)

결합 규칙은 meta 분할에서 학습하는데 meta가 47개뿐인 물성이 있어 규칙이 잡음을 학습한다. train 분할을 함께 쓰면 표본이 여덟 배로 늘지만, 모델이 train 분자를 이미 학습했으므로 그 예측은 외운 값이라 쓸 수 없다. cv_fold로 다섯 조각을 나눠 조각마다 따로 미세조정한 뒤, 각 분자를 자기가 빠진 모델로 예측한다.

물성 하나당 학습이 열 번(5폴드 × 정규·증강)이라 로컬 MPS로는 물성당 수십 분이 걸린다. GPU에서는 몇 분이면 끝난다.

실행 전 확인

- 런타임 유형을 GPU로 바꾼다. 메뉴에서 런타임 → 런타임 유형 변경 → T4 GPU
- 드라이브의 `Conference_2026` 폴더에 접근 권한이 있어야 한다

이미 끝난 물성은 `--resume`으로 건너뛴다. 로컬에서 처리한 것이 드라이브에 있으면 그대로 인정된다.

In [ ]:
import torch

assert torch.cuda.is_available(), "GPU 런타임이 아니다. 런타임 → 런타임 유형 변경에서 T4 GPU를 고른다"
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/Conference_2026"
REPO = f"{DRIVE}/Juhyeong"
print(open(f"{REPO}/scripts/score_train_oof_chemberta.py").readline())

In [ ]:
!pip install -q rdkit transformers

In [ ]:
# 입력을 로컬 디스크로 복사한다. 드라이브에서 직접 읽으면 작은 파일이 많아 느리다.
import shutil
from pathlib import Path

LOCAL = Path("/content/work")
LOCAL.mkdir(exist_ok=True)

for name, source in [
    ("splits", f"{DRIVE}/Yoonsoo/processed"),
    ("variants_train", f"{REPO}/data/processed/variants_train"),
]:
    target = LOCAL / name
    if not target.exists():
        shutil.copytree(source, target, ignore=shutil.ignore_patterns("_cache", "reports", "*.gsheet"))
    print(name, len(list(target.iterdir())), "개 폴더")

shutil.copy(f"{REPO}/scripts/score_train_oof_chemberta.py", LOCAL)
shutil.copy(f"{REPO}/scripts/dataset_repairs.py", LOCAL)

# 이미 끝난 물성을 로컬로 가져와 --resume이 인식하게 한다.
OUT = LOCAL / "train_oof_chemberta"
DRIVE_OUT = Path(f"{REPO}/data/processed/scores_role4/train_oof_chemberta")
if DRIVE_OUT.exists() and not OUT.exists():
    shutil.copytree(DRIVE_OUT, OUT)
done = sorted(p.name for p in OUT.iterdir() if p.is_dir() and p.name != "_summary") if OUT.exists() else []
print("이미 완료:", done)

In [ ]:
DATASETS = [
    "dili", "herg", "hia_hou", "bioavailability_ma",
    "cyp2d6_substrate_carbonmangels", "cyp2c9_substrate_carbonmangels",
    "cyp3a4_substrate_carbonmangels", "half_life_obach", "caco2_wang",
    "clearance_hepatocyte_az", "vdss_lombardo", "clearance_microsome_az",
    "pgp_broccatelli", "ppbr_az", "bbb_martins",
]

!cd /content/work && python score_train_oof_chemberta.py \
  --processed-dir splits \
  --variants-dir variants_train \
  --out-dir train_oof_chemberta \
  --device cuda --resume \
  --datasets {' '.join(DATASETS)} 2>&1 | grep -vE "newly initialized|down-stream|Warning"

In [ ]:
# 산출물 검증. 행 수가 원본 train 및 변형 수와 맞는지 확인한다.
import pandas as pd

rows = []
for dataset in DATASETS:
    origin = pd.read_csv(OUT / dataset / "origin_predictions_chemberta_oof.csv")
    variant = pd.read_csv(OUT / dataset / "variant_predictions_chemberta_oof.csv")
    splits = pd.read_csv(LOCAL / "splits" / dataset / "splits.csv", low_memory=False)
    expected = int(splits["split"].eq("train").sum())
    missing = int(
        origin[["pred_chemberta_regular", "pred_chemberta_augmented"]].isna().sum().sum()
        + variant[["pred_chemberta_regular", "pred_chemberta_augmented"]].isna().sum().sum()
    )
    rows.append({
        "dataset": dataset, "원본": len(origin), "기대": expected,
        "변형": len(variant), "결측": missing,
        "정상": len(origin) == expected and missing == 0,
    })

check = pd.DataFrame(rows)
print(check.to_string(index=False))
print()
print("모두 정상" if check["정상"].all() else "확인 필요한 물성이 있다")

In [ ]:
# 검증을 통과한 경우에만 드라이브로 되돌린다.
assert check["정상"].all(), "검증 실패. 위 표를 확인한다"

if DRIVE_OUT.exists():
    shutil.rmtree(DRIVE_OUT)
shutil.copytree(OUT, DRIVE_OUT)
print("드라이브 반영 완료:", DRIVE_OUT)
print(sorted(p.name for p in DRIVE_OUT.iterdir()))